# Recap Studio — Gemini Voice Clone (Colab)

ဒီ notebook က Recap Studio backend နဲ့ **တူညီတဲ့** Gemini Voices API (`POST /v1beta/voices`) ကို သုံးပါတယ်။

## လုပ်ရမယ့်အဆင့်
1. အောက်က cell တွေကို အပေါ်ကနေ အောက်အထိ **Run**
2. Gemini API key ထည့်
3. Reference + Consent အသံ ၂ ခု တင်
4. ရတဲ့ `voice_...` ID ကို Recap Studio → အသံ Clone → **Colab Voice ID ထည့်မည်** ထဲ ကူးထည့်

### Consent စာသား (မြန်မာ — စကားလုံးအတိုင်း ဖတ်ပါ)
> ကျွန်ုပ်သည် ဤအသံ၏ပိုင်ရှင်ဖြစ်ပြီး Google Cloud ကိုအသုံးပြုခြင်းဖြင့် ကျွန်ုပ်၏အသံ၏ ပေါင်းစပ်ပုံစံတစ်ခု ဖန်တီးရန် သဘောတူပါသည်။


In [ ]:
# 1) Install tools
!pip -q install -U pydub requests
!apt-get -qq install -y ffmpeg > /dev/null

from IPython.display import display, Markdown
display(Markdown("✅ Install OK — နောက် cell ဆက် Run ပါ"))


In [ ]:
# 2) Gemini API key
# Colab Secrets ရှိရင် အလိုအလျောက် သုံးမယ်။ မရှိရင် paste လုပ်ပါ။
import os
from getpass import getpass

API_KEY = ""
try:
    from google.colab import userdata
    API_KEY = (userdata.get("GEMINI_API_KEY") or "").strip()
except Exception:
    API_KEY = ""

if not API_KEY:
    API_KEY = getpass("Gemini API key paste လုပ်ပါ (မပေါ်ပါ): ").strip()

if not API_KEY:
    raise SystemExit("❌ API key မရှိပါ။ https://aistudio.google.com/apikey မှာ ယူပြီး ပြန် Run ပါ")

os.environ["GEMINI_API_KEY"] = API_KEY
print("✅ API key loaded (", len(API_KEY), "chars )")


In [ ]:
# 3) Upload reference + consent audio
from google.colab import files

print("① Reference အသံ ရွေးပါ (~10–30 စက္ကန့် ပုံမှန်ပြောသံ)")
ref_up = files.upload()
if not ref_up:
    raise SystemExit("❌ Reference အသံ မတင်ရသေးပါ")

print("② Consent အသံ ရွေးပါ (အပေါ်က မြန်မာစာသားကို စကားလုံးအတိုင်း ဖတ်ထားသောဖိုင်)")
con_up = files.upload()
if not con_up:
    raise SystemExit("❌ Consent အသံ မတင်ရသေးပါ")

REF_PATH = next(iter(ref_up))
CON_PATH = next(iter(con_up))
print("reference:", REF_PATH)
print("consent:", CON_PATH)


In [ ]:
# 4) Convert to 24kHz mono WAV
from pathlib import Path
from pydub import AudioSegment

def to_wav_24k(src: str, dest: str):
    audio = AudioSegment.from_file(src)
    audio = audio.set_channels(1).set_frame_rate(24000).set_sample_width(2)
    audio.export(dest, format="wav")
    sec = len(audio) / 1000.0
    print(f"{src} → {dest} ({sec:.1f}s)")
    return dest, sec

ref_wav, ref_sec = to_wav_24k(REF_PATH, "reference_24k.wav")
con_wav, con_sec = to_wav_24k(CON_PATH, "consent_24k.wav")

if ref_sec < 8:
    raise SystemExit("❌ Reference အသံ တိုလွန်းပါတယ်။ အနည်းဆုံး ~10 စက္ကန့် ထားပါ")
if con_sec < 3:
    raise SystemExit("❌ Consent အသံ တိုလွန်းပါတယ်။ စာသားအပြည့် ဖတ်ထားပါ")
print("✅ Audio ready")


In [ ]:
# 5) Clone voice (same API as Recap Studio backend)
import base64
import json
import requests

DISPLAY_NAME = "ThuHtooSan"  # နာမည်ပြင်ချင်ရင် ဒီနေရာ

def b64file(path: str) -> str:
    return base64.b64encode(Path(path).read_bytes()).decode("utf-8")

payload = {
    "store": True,
    "voice": {
        "type": "replicated",
        "display_name": DISPLAY_NAME,
        "replicated": {
            "source_audio": {"mime_type": "audio/wav", "data": b64file(ref_wav)},
            "consent_audio": {"mime_type": "audio/wav", "data": b64file(con_wav)},
        },
    },
}

print("Gemini ကို ပို့နေပါတယ်… (၁–၂ မိနစ် ကြာနိုင်ပါတယ်)")
res = requests.post(
    "https://generativelanguage.googleapis.com/v1beta/voices",
    headers={
        "content-type": "application/json",
        "x-goog-api-key": API_KEY,
    },
    data=json.dumps(payload),
    timeout=180,
)

body = {}
try:
    body = res.json()
except Exception:
    body = {"raw": res.text[:500]}

if not res.ok:
    msg = ""
    if isinstance(body, dict):
        msg = (body.get("error") or {}).get("message") or str(body)[:400]
    else:
        msg = str(body)[:400]
    lower = msg.lower()
    if "consent" in lower or "transcript" in lower or "does not match" in lower or "invalid_argument" in lower:
        raise SystemExit(
            "❌ Consent အသံက စာသားနဲ့ မကိုက် / Reference မရှင်းပါ။\n"
            "မြန်မာ Consent စာသားကို စကားလုံးအတိုင်း ပြန်ဖတ်ပြီး Consent ဖိုင်အသစ်တင်ပါ။\n\n"
            f"Detail: {msg[:250]}"
        )
    raise SystemExit(f"❌ Clone failed HTTP {res.status_code}: {msg[:300]}")

VOICE_ID = body.get("id") or body.get("key")
if not VOICE_ID:
    raise SystemExit(f"❌ Voice ID မရပါ။ response: {json.dumps(body)[:400]}")

print("✅ Clone OK")
print("VOICE_ID =", VOICE_ID)
print("ဒီ ID ကို Recap Studio → အသံ Clone → Colab Voice ID ထည့်မည် ထဲ ကူးထည့်ပါ")


In [ ]:
# 6) Optional test speak
import wave
import struct

text = "မင်္ဂလာပါ။ ဒါက ကျွန်တော့် clone အသံ စမ်းသပ်မှုပါ။"
tts_payload = {
    "contents": [{"role": "user", "parts": [{"text": text}]}],
    "generationConfig": {
        "responseModalities": ["AUDIO"],
        "speechConfig": {"voiceConfig": {"voice": VOICE_ID}},
    },
}
tts = requests.post(
    "https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash-tts:generateContent",
    headers={"content-type": "application/json", "x-goog-api-key": API_KEY},
    data=json.dumps(tts_payload),
    timeout=120,
)
tj = tts.json()
part = (((tj.get("candidates") or [{}])[0].get("content") or {}).get("parts") or [{}])[0]
inline = part.get("inlineData") or part.get("inline_data") or {}
data_b64 = inline.get("data")
mime = inline.get("mimeType") or inline.get("mime_type") or ""

if not tts.ok or not data_b64:
    print("Test speak skipped:", (tj.get("error") or {}).get("message") or tts.status_code)
else:
    raw = base64.b64decode(data_b64)
    out = Path("clone_test.wav")
    # Gemini sometimes returns raw PCM; wrap as WAV if needed
    if raw[:4] == b"RIFF":
        out.write_bytes(raw)
    else:
        with wave.open(str(out), "wb") as w:
            w.setnchannels(1)
            w.setsampwidth(2)
            w.setframerate(24000)
            w.writeframes(raw)
    print("Saved", out)
    from IPython.display import Audio, display
    display(Audio(str(out)))
